# DEBUG: StandUp4AI Feature Extraction

Step-by-step debugging.

In [ ]:
# Mount and find files
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# Try to find the standup4ai folder
paths_to_try = [
    '/content/drive/MyDrive/standup4ai',
    '/content/drive/Shareddrives/standup4ai',
    '/content/drive/MyDrive/standup4ai/audio',
]

BASE = None
for p in paths_to_try:
    if os.path.exists(p):
        BASE = p
        print(f'Found at: {p}')
        break

if BASE is None:
    # List what exists
    print('Checking /content/drive...')
    for root, dirs, files in os.walk('/content/drive'):
        for d in dirs:
            if 'standup' in d.lower() or 'laugh' in d.lower():
                print(f'  Found: {os.path.join(root, d)}')
        if len(list(os.walk('/content/drive'))) > 100:
            break

print(f'BASE = {BASE}')
if BASE:
    AUDIO_DIR = os.path.join(BASE, 'audio')
    LABELS_DIR = os.path.join(BASE, 'labels')
    print(f'Audio dir: {AUDIO_DIR}, exists={os.path.exists(AUDIO_DIR)}')
    print(f'Labels dir: {LABELS_DIR}, exists={os.path.exists(LABELS_DIR)}')


In [ ]:
# List audio files
if BASE:
    audio_files = [f for f in os.listdir(AUDIO_DIR) if f.endswith('.m4a') or f.endswith('.mp3')]
    print(f'Audio files found: {len(audio_files)}')
    print(f'Sample: {sorted(audio_files)[:5]}')
    
    label_files = [f for f in os.listdir(LABELS_DIR) if f.endswith('.csv')]
    print(f'Label files found: {len(label_files)}')
    print(f'Sample: {sorted(label_files)[:5]}')
    
    # Find overlap
    audio_vids = {f.replace('.m4a','').replace('.mp3','') for f in audio_files}
    label_vids = {f.replace('.csv','') for f in label_files}
    overlap = sorted(audio_vids & label_vids)
    print(f'Overlap (have both): {len(overlap)}')
    print(f'Sample: {overlap[:5]}')


In [ ]:
# Test audio loading with librosa
import subprocess
subprocess.run(['pip', 'install', '-q', 'librosa'], check=True, timeout=30)

import librosa
import numpy as np

if BASE and overlap:
    test_vid = overlap[0]
    audio_path = os.path.join(AUDIO_DIR, f'{test_vid}.m4a')
    print(f'Testing audio: {audio_path}')
    print(f'File size: {os.path.getsize(audio_path)} bytes')
    
    # Try loading first 10 seconds
    try:
        y, sr = librosa.load(audio_path, sr=22050, duration=10.0)
        print(f'Loaded! y.shape={y.shape}, sr={sr}')
        print(f'Duration: {len(y)/sr:.1f}s')
    except Exception as e:
        print(f'ERROR loading: {e}')
        
    # Try with audioread
    try:
        y2, sr2 = librosa.load(audio_path, sr=22050, duration=3.0, res_type='audioread')
        print(f'audioread OK: {y2.shape}')
    except Exception as e:
        print(f'audioread ERROR: {e}')


In [ ]:
# Test label loading
import pandas as pd

if BASE and overlap:
    test_vid = overlap[0]
    label_path = os.path.join(LABELS_DIR, f'{test_vid}.csv')
    print(f'Testing labels: {label_path}')
    print(f'File size: {os.path.getsize(label_path)} bytes')
    
    try:
        df = pd.read_csv(label_path)
        print(f'Loaded! shape={df.shape}')
        print(f'Columns: {list(df.columns)}')
        print(f'First 3 rows:')
        print(df.head(3))
        print(f'Label values: {df["label"].value_counts().to_dict()}')
    except Exception as e:
        print(f'ERROR: {e}')
        # Try reading raw
        with open(label_path, 'r') as f:
            lines = f.readlines()[:5]
        print(f'Raw lines: {lines}')


In [ ]:
# Test feature extraction on one segment
if BASE and overlap:
    test_vid = overlap[0]
    audio_path = os.path.join(AUDIO_DIR, f'{test_vid}.m4a')
    label_path = os.path.join(LABELS_DIR, f'{test_vid}.csv')
    
    df = pd.read_csv(label_path)
    row = df.iloc[0]
    t0, t1 = float(row['t0']), float(row['t1'])
    print(f'Extracting segment: {t0:.2f}s - {t1:.2f}s')
    
    dur = min(t1 - t0, 10.0)
    print(f'Duration: {dur:.2f}s')
    
    try:
        y, sr = librosa.load(audio_path, sr=22050, offset=t0, duration=dur, mono=True)
        print(f'Audio loaded: {y.shape}, sr={sr}')
    except Exception as e:
        print(f'Audio load ERROR: {e}')
        y = None
    
    if y is not None and len(y) > 0:
        # F0 extraction
        try:
            f0, voiced_flag, voiced_probs = librosa.pyin(y, fmin=80, fmax=500, sr=sr, hop_length=512)
            f0 = np.nan_to_num(f0, nan=0)
            print(f'F0: mean={np.mean(f0):.1f}Hz, voiced_rate={np.mean(voiced_flag):.2f}')
        except Exception as e:
            print(f'F0 ERROR: {e}')
        
        # RMS
        hop = 512
        rms = librosa.feature.rms(y=y, hop_length=hop)[0]
        print(f'RMS: mean={np.mean(rms):.4f}, max={np.max(rms):.4f}')
        
        # ZCR
        zcr = librosa.feature.zero_crossing_rate(y, hop_length=hop)[0]
        print(f'ZCR: mean={np.mean(zcr):.4f}')
        
        print('Feature extraction SUCCESS!')
    else:
        print('Audio was empty!')


In [ ]:
# If all works, run full extraction
if BASE and overlap:
    print(f'Running full extraction on {len(overlap)} videos...')
    
    X_all, y_all, vids_all = [], [], []
    
    for i, vid in enumerate(overlap):
        audio_path = os.path.join(AUDIO_DIR, f'{vid}.m4a')
        label_path = os.path.join(LABELS_DIR, f'{vid}.csv')
        
        if not os.path.exists(audio_path) or not os.path.exists(label_path):
            continue
        
        df = pd.read_csv(label_path)
        
        for _, seg in df.iterrows():
            try:
                t0, t1 = float(seg['t0']), float(seg['t1'])
                dur = min(t1 - t0, 10.0)
                if dur < 0.1:
                    continue
                
                y, sr = librosa.load(audio_path, sr=22050, offset=t0, duration=dur, mono=True)
                if len(y) < sr * 0.1:
                    continue
                
                f0, voiced_flag, _ = librosa.pyin(y, fmin=80, fmax=500, sr=sr, hop_length=512)
                f0 = np.nan_to_num(f0, nan=0)
                
                rms = librosa.feature.rms(y=y, hop_length=512)[0]
                zcr = librosa.feature.zero_crossing_rate(y, hop_length=512)[0]
                
                feat = np.array([
                    np.mean(f0), np.std(f0), np.mean(voiced_flag),
                    np.mean(rms), np.std(rms), np.max(rms),
                    np.mean(zcr), np.std(zcr),
                    len(y)/sr
                ], dtype=np.float32)
                
                X_all.append(feat)
                y_all.append(1 if seg['label'].strip() == 'risa' else 0)
                vids_all.append(vid)
                
            except Exception as e:
                pass  # Skip failed segments
        
        if (i + 1) % 10 == 0:
            print(f'  {i+1}/{len(overlap)} videos, {len(X_all)} samples...')
    
    print(f'\n=== FINAL ===')
    print(f'Total samples: {len(X_all)}')
    print(f'Positive: {sum(y_all)} ({sum(y_all)/len(y_all):.1%})')
    print(f'Videos: {len(set(vids_all))}')
    
    if len(X_all) > 0:
        X = np.array(X_all)
        y = np.array(y_all)
        print(f'Feature dim: {X.shape[1]}')
        print('READY FOR TRAINING!')
